# Bronze Layer Ingestion

Este notebook carga los datos desde el volumen (landing) hacia tablas Delta en la capa Bronze.

## Origen de datos
/Volumes/proyecto_smart_claims/landing/volumen_landing/

## Tablas generadas
- bronze.customers
- bronze.policies
- bronze.claims
- bronze.telematics
- bronze.training_images
- bronze.claim_images
- bronze.claim_images_metadata

**Importamos librerias**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

**Configuración**

In [0]:
CATALOG = "proyecto_smart_claims"
LANDING_SCHEMA = "landing"
VOLUME_NAME = "volumen_landing"

customers = F"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/customers/customers.csv"

policies = F"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/policies/policies.csv"

claims = F"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/claims/claims.csv"

telematics = F"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/telematics"

training_imgs = F"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/training_imgs"

claim_images = F"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/claim_images"

claim_images_metadata = F"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/claim_images_metadata/image_metadata.csv"

spark = SparkSession.builder.getOrCreate()

**Leer CSV**

In [0]:
#Leemos customers
customers = (
   spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/proyecto_smart_claims/landing/volumen_landing/customers/customers.csv")     
 )
 
policies= (
   spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/proyecto_smart_claims/landing/volumen_landing/policies/policies.csv")     
 )

  #Leemos claims
claims = (
   spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/proyecto_smart_claims/landing/volumen_landing/claims/claims.csv")     
 )

  #Leemos claim_images_metadata
claim_images_metadata = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/proyecto_smart_claims/landing/volumen_landing/claim_images_metadata/image_metadata.csv")     
 )

  #Leemos training_images
training_imgs = (
    spark.read.format("binaryFile")\
    .load("/Volumes/proyecto_smart_claims/landing/volumen_landing/training_imgs/") 
 )

 #Leemos claim_images
claim_images = (
    spark.read.format("binaryFile")\
    .load("/Volumes/proyecto_smart_claims/landing/volumen_landing/claim_images/") 
 )

#Leemos telematics
telematics= (
    spark.read
    .parquet("/Volumes/proyecto_smart_claims/landing/volumen_landing/telematics/") 
 )

**Escribir en Bronze (Delta)**

In [0]:
#Guardar tablas Bronze

customers.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.bronze.customers")

policies.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.bronze.policies")

claims.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.bronze.claims")

claim_images_metadata.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.bronze.claim_images_metadata")

training_imgs.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.bronze.training_imgs")

claim_images.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.bronze.claim_images")

telematics.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{CATALOG}.bronze.telematics")

In [0]:
print("Bronze ingestion completada")
print(f"Tablas creadas: {CATALOG}.bronze.customers, policies, claims, claim_images_metadata,training_imgs, claim_images, telematics")